# 03: Gateway with Cedar authorization policies

Combine a **Bedrock Managed Knowledge Base (BMKB)** with an **AgentCore Gateway** and a
**Cedar Policy Engine** to add per-request, policy-as-code authorization on top of retrieval.

The Gateway already hides the KB ID from agents (see
[01-bmkb-with-agentcore-gateway.ipynb](01-end-to-end-example-with-ac-gateway/01-bmkb-with-agentcore-gateway.ipynb)).
Here we add a **Cedar Policy Engine** so the Gateway also decides *whether a caller is allowed
to invoke it at all*. That decision runs on every request, gets logged, and does not touch
the KB.

### What you will build

1. **Create a BMKB** and ingest documents (via the `ManagedKnowledgeBase` utility)
2. **Create a Cedar Policy Engine** through the AgentCore control plane
3. **Create a Cedar-enabled Gateway** in `LOG_ONLY` mode (evaluate and log, don't block)
4. **Attach the KB as a Gateway Target** and write a Cedar **permit** policy
5. **Test authorized access.** A permitted caller retrieves through the Gateway
6. **Flip to `ENFORCE` and add a `forbid` policy** to update authorization dynamically
7. **Test denied access.** The same caller is now blocked with JSON-RPC `-32002`
8. **Clean up** every resource

### Architecture

```
Agent (IAM) ──► Gateway ──► Cedar Policy Engine ──► KB Target ──► Managed KB
                  │              │
                  │              └── permit / forbid based on principal + resource
                  └── LOG_ONLY: evaluate + log, never block
                      ENFORCE:  evaluate + block denied calls (-32002)
```

### What Cedar controls (and what it doesn't)

| Cedar controls | Cedar does **not** control |
|---|---|
| Whether a caller may invoke a Gateway (resource-level) | Which documents are returned |
| Resource-level permit and deny, as code | Document-level or metadata-based filtering |

For document-level scoping, combine this pattern with metadata filters
(see [Pattern 2: Metadata Filters](../04-security-and-access-controls/notebooks/02-metadata-filters.ipynb)).

> **Relationship to Pattern 4.** The Cedar mechanics here (policy engine, permit/forbid,
> the SigV4 MCP client) mirror [Pattern 4: Gateway + Cedar](../04-security-and-access-controls/notebooks/04-gateway-cedar.ipynb).
> That notebook is a security-pattern reference (Pattern 4) built on raw boto3. This one is a
> use-case walkthrough built on the `ManagedKnowledgeBase` utility, so the KB lifecycle, the
> cleanup guard, and the propagation-aware deny test differ. Treat Pattern 4 as the canonical
> source for the Cedar details.

## Prerequisites

- AWS credentials with permissions for **Bedrock**, **IAM**, **S3**, and **AgentCore**
  (`bedrock-agentcore-control`: Gateway, Policy Engine, Cedar policies)
- Model access enabled for the **managed default embedding** model
- Familiarity with the [Cedar policy language](https://www.cedarpolicy.com/)
- Python 3.10+ with `boto3`, `mcp`, `httpx` (installed below)
- **Kernel:** Select `Python 3`

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --quiet
# mcp + httpx power the SigV4 MCP client used in the permit/deny tests (Steps 9 & 11).
# Pin mcp>=1.9: streamable_http_client(url, http_client=...) — passing a pre-built
# httpx client for SigV4 signing — requires the newer transport signature; older
# 1.x releases lack the http_client parameter. (Verified against mcp 1.28.1.)
%pip install "mcp>=1.9" httpx --quiet

In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Step 1: Configuration

Set the region and resource names. A time-based suffix keeps resource names unique across runs.

In [ ]:
import boto3
import sys
import time
import json

sys.path.insert(0, "..")   # repo root (bmkb-code-samples) is one level up

session = boto3.session.Session()
region = session.region_name or "us-west-2"
account_id = boto3.client("sts").get_caller_identity()["Account"]

suffix = time.strftime("%Y%m%d%H%M%S", time.localtime())[-7:]

# ── Configuration ─────────────────────────────────────────────────────
knowledge_base_name = f"bmkb-cedar-{suffix}"
bucket_name         = f"bedrock-bmkb-cedar-{suffix}-{account_id}"
gateway_name        = f"bmkb-cedar-gw-{suffix}"
# Policy Engine names must match ^[A-Za-z][A-Za-z0-9_]*$ (letters/digits/underscore,
# no hyphens) — unlike KB/gateway names, which allow hyphens.
policy_engine_name  = f"bmkb_cedar_pe_{suffix}"

# Gateway IAM role — a Cedar-enabled gateway needs extra permissions beyond a
# plain retrieve role (see Step 4). Auto-created below.
gateway_role_name   = f"AmazonBedrockGatewayCedarRole_{suffix}"
gateway_policy_name = f"AmazonBedrockGatewayCedarPolicy_{suffix}"

# Managed default embedding (no extra cost). Set an embedding model id for a custom one.
embedding_model = None

# Raw AgentCore control-plane client — the utility manages the KB; we drive the
# Policy Engine, Cedar-enabled gateway, target, and policies directly since the
# utility's create_gateway() does not accept a policyEngineConfiguration.
ac = session.client("bedrock-agentcore-control", region_name=region)

print(f"boto3:    {boto3.__version__}")
print(f"Region:   {region}")
print(f"Account:  {account_id}")
print(f"KB:       {knowledge_base_name}")
print(f"Bucket:   {bucket_name}")
print(f"Gateway:  {gateway_name}")
print(f"Engine:   {policy_engine_name}")

## Step 2: Create the S3 bucket and upload documents

We ingest two synthetic documents: Octank Financial's 10-K annual report and a U.S. tornado
forecasting report. In production this is your own corpus.

In [ ]:
s3 = boto3.client("s3", region_name=region)

try:
    s3.head_bucket(Bucket=bucket_name)
    print(f"Bucket already exists: {bucket_name}")
except Exception:
    print(f"Creating bucket: {bucket_name}")
    if region == "us-east-1":
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region},
        )

for fname in ("octank_financial_10K.pdf", "tornadoes_report.pdf"):
    path = f"../synthetic_dataset/{fname}"
    print(f"Uploading {fname}")
    s3.upload_file(path, bucket_name, fname)

print("Documents uploaded.")

## Step 3: Create the Bedrock managed knowledge base and ingest

The `ManagedKnowledgeBase` utility handles the full KB lifecycle: the IAM execution role, the
managed vector store, the S3 data source via the managed connector, and ingestion.

In [ ]:
from utils.managed_knowledge_base import ManagedKnowledgeBase

kb = ManagedKnowledgeBase(
    kb_name=knowledge_base_name,
    bucket_name=bucket_name,
    embedding_model=embedding_model,
    enable_logging=True,
    region_name=region,
    suffix=suffix,
)

print(f"\nKB ID: {kb.kb_id}")
print(f"DS ID: {kb.ds_id}")

In [ ]:
time.sleep(30)          # let the data source settle before ingesting
kb.start_ingestion_job()

In [ ]:
# Sanity check — direct SDK retrieve (bypasses the Gateway; Cedar never sees this).
# We only confirm the KB itself works before adding the authorization layer.
resp = kb.retrieve("What are Octank's key financial results?", num_results=3)
print("=== Direct SDK Retrieve (pre-Gateway) ===")
for i, r in enumerate(resp.get("retrievalResults", []), 1):
    print(f"  {i}. score={r['score']:.4f} | {r['content']['text'][:100]}...")

## Step 4: Create the Cedar-aware Gateway IAM role

A Cedar-enabled Gateway needs more than a plain retrieve role. Beyond `bedrock:Retrieve`
on the KB, the role must let the Gateway **read the Policy Engine** and invoke the Cedar
**`Authorize*`** action family (evaluated against both the policy-engine and gateway
resources). Without these, gateway creation or evaluation fails.

In [ ]:
iam = boto3.client("iam")

gw_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
        "Action": "sts:AssumeRole",
        "Condition": {"StringEquals": {"aws:SourceAccount": account_id}},
    }],
}

gw_permission_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "KnowledgeBaseRetrieve",
            "Effect": "Allow",
            "Action": [
                "bedrock:Retrieve",
                "bedrock:GetKnowledgeBase",
                "bedrock:ListKnowledgeBases",
            ],
            "Resource": f"arn:aws:bedrock:{region}:{account_id}:knowledge-base/*",
        },
        {
            # Read the Policy Engine — checked when the gateway is created.
            "Sid": "PolicyEngineRead",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetPolicyEngine",
                "bedrock-agentcore:GetPolicyEngineSummary",
            ],
            "Resource": f"arn:aws:bedrock-agentcore:{region}:{account_id}:policy-engine/*",
        },
        {
            # Cedar evaluation — the engine calls a family of authorize actions,
            # checked against BOTH the policy-engine and gateway resources.
            "Sid": "PolicyEngineAuthorize",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:Authorize*",
                "bedrock-agentcore:PartiallyAuthorize*",
            ],
            "Resource": [
                f"arn:aws:bedrock-agentcore:{region}:{account_id}:policy-engine/*",
                f"arn:aws:bedrock-agentcore:{region}:{account_id}:gateway/*",
            ],
        },
    ],
}

try:
    role = iam.create_role(
        RoleName=gateway_role_name,
        AssumeRolePolicyDocument=json.dumps(gw_trust_policy),
        Description=f"Cedar-enabled gateway role for {knowledge_base_name}",
        MaxSessionDuration=3600,
    )
    print(f"Created role: {gateway_role_name}")
except iam.exceptions.EntityAlreadyExistsException:
    role = iam.get_role(RoleName=gateway_role_name)
    print(f"Role already exists: {gateway_role_name}")
gateway_role_arn = role["Role"]["Arn"]

try:
    policy_arn = iam.create_policy(
        PolicyName=gateway_policy_name,
        PolicyDocument=json.dumps(gw_permission_policy),
    )["Policy"]["Arn"]
    print(f"Created policy: {gateway_policy_name}")
except iam.exceptions.EntityAlreadyExistsException:
    policy_arn = f"arn:aws:iam::{account_id}:policy/{gateway_policy_name}"
    print(f"Policy already exists: {gateway_policy_name}")

iam.attach_role_policy(RoleName=gateway_role_name, PolicyArn=policy_arn)

print("Waiting for IAM propagation...")
time.sleep(30)
print(f"Gateway role ARN: {gateway_role_arn}")

## Step 5: Create the Cedar Policy Engine

The Policy Engine holds your Cedar policies and evaluates them for every Gateway request.
Create it first, because the Gateway references it by ARN.

In [ ]:
pe_response = ac.create_policy_engine(name=policy_engine_name)
pe_id  = pe_response["policyEngineId"]
pe_arn = pe_response["policyEngineArn"]
print(f"Policy Engine: {pe_id}")
print(f"Policy Engine ARN: {pe_arn}")

for _ in range(12):
    pe = ac.get_policy_engine(policyEngineId=pe_id)
    if pe["status"] == "ACTIVE":
        break
    time.sleep(5)
print(f"Status: {pe['status']}")

## Step 6: Create the Cedar-enabled Gateway (`LOG_ONLY`)

Create the Gateway with a `policyEngineConfiguration` pointing at the engine. We start in
**`LOG_ONLY`** mode, where Cedar evaluates every request and logs the decision but never
blocks. That is the safe way to validate policies before enforcing them.

> The `ManagedKnowledgeBase.create_gateway()` helper doesn't accept a policy engine, so we
> call `bedrock-agentcore-control` directly here.

In [ ]:
gw_response = ac.create_gateway(
    name=gateway_name,
    roleArn=gateway_role_arn,
    protocolType="MCP",
    authorizerType="AWS_IAM",
    policyEngineConfiguration={
        "arn": pe_arn,
        "mode": "LOG_ONLY",   # evaluate + log, don't block
    },
)
gw_id = gw_response["gatewayId"]
print(f"Gateway: {gw_id}")

gw_url = None
for _ in range(24):
    gw = ac.get_gateway(gatewayIdentifier=gw_id)
    if gw["status"] == "READY":
        gw_url = gw.get("gatewayUrl", "N/A")
        break
    time.sleep(5)
print(f"Status: {gw['status']}")
print(f"Gateway URL: {gw_url}")

## Step 7: Write a Cedar **permit** policy

Cedar policies use the `AgentCore::Gateway` entity type; the resource is the Gateway ARN.
For a policy that will actually enforce later, the **resource must be constrained** to a
specific Gateway. A bare wildcard resource is rejected at create time.

```cedar
permit(principal, action, resource == AgentCore::Gateway::"<gateway-arn>");
```

This permits all callers to invoke this Gateway. (To scope to one caller, constrain
`principal` instead of leaving it unbound.)

In [ ]:
gw_arn = f"arn:aws:bedrock-agentcore:{region}:{account_id}:gateway/{gw_id}"

permit_statement = f'permit(principal, action, resource == AgentCore::Gateway::"{gw_arn}");'
print(f"Cedar policy:\n  {permit_statement}")

# Policy names must be unique per engine; derive one from gw_id so re-runs don't collide.
permit_response = ac.create_policy(
    policyEngineId=pe_id,
    name=f"permit_gateway_{gw_id.split('-')[-1]}",
    definition={"cedar": {"statement": permit_statement}},
    validationMode="IGNORE_ALL_FINDINGS",
)
permit_policy_id = permit_response["policyId"]

for _ in range(12):
    p = ac.get_policy(policyEngineId=pe_id, policyId=permit_policy_id)
    if p["status"] == "ACTIVE":
        break
    time.sleep(5)
print(f"Permit policy: {permit_policy_id} [{p['status']}]")

## Step 8: Attach the KB as a Gateway target

The target wires the Gateway to the KB via the `bedrock-knowledge-bases` connector. Agents
call the target's tool (`kb-retrieve___Retrieve`) over MCP, and they never see the KB ID.

In [ ]:
target_response = ac.create_gateway_target(
    gatewayIdentifier=gw_id,
    name="kb-retrieve",
    targetConfiguration={
        "mcp": {
            "connector": {
                "source": {"connectorId": "bedrock-knowledge-bases"},
                "configurations": [{
                    "name": "Retrieve",
                    # Tool description exposed to the agent over MCP — what the LLM
                    # reads to decide when to call this KB.
                    "description": (
                        "Search two corporate documents: (1) Octank Financial's 10-K annual "
                        "report — financial statements, asset/liability schedules, exhibits, and "
                        "investor disclosures; and (2) a U.S. tornado background & forecasting "
                        "report — where tornadoes form, annual frequency (~1,200/yr), and NOAA data."
                    ),
                    "parameterValues": {
                        "knowledgeBaseId": kb.kb_id,
                        "retrievalConfiguration": {
                            "managedSearchConfiguration": {"numberOfResults": 5}
                        },
                    },
                }],
            }
        }
    },
    credentialProviderConfigurations=[
        {"credentialProviderType": "GATEWAY_IAM_ROLE"}
    ],
)
target_id = target_response["targetId"]
print(f"Target: {target_id}")

for _ in range(12):
    t = ac.get_gateway_target(gatewayIdentifier=gw_id, targetId=target_id)
    if t["status"] == "READY":
        break
    time.sleep(5)
print(f"Target status: {t['status']}")

## Step 9: Test authorized access (permit + `LOG_ONLY`)

We retrieve **through the Gateway**, which is the only path Cedar sees. The `kb.retrieve()`
call in Step 3 hits Bedrock directly by KB ID and bypasses the Gateway entirely.

An `AWS_IAM` Gateway requires every request to be **SigV4-signed**. The MCP client talks over
`httpx`, which has no built-in SigV4, so we wrap botocore's signer in a small `httpx.Auth`
and hand it to the client. With the permit policy in `LOG_ONLY` mode, the call succeeds.

In [ ]:
import httpx
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from mcp.shared.exceptions import McpError


class SigV4HTTPXAuth(httpx.Auth):
    """httpx auth handler that SigV4-signs each request for bedrock-agentcore."""
    requires_request_body = True   # we must see the body to sign it

    def __init__(self, credentials, service, region):
        self._credentials, self._service, self._region = credentials, service, region

    def auth_flow(self, request):
        aws_req = AWSRequest(
            method=request.method,
            url=str(request.url),
            data=request.content,
            headers=dict(request.headers),
        )
        SigV4Auth(self._credentials, self._service, self._region).add_auth(aws_req)
        request.headers.update(dict(aws_req.headers))   # copy signed headers back
        yield request


sigv4 = SigV4HTTPXAuth(session.get_credentials(), "bedrock-agentcore", region)

# The forbid policy in Step 10 also hides the tool from tools/list, so resolve the
# tool name now (while permitted) and reuse it in Step 11. Naming convention is
# `<target-name>___<tool-name>`.
tool_name = "kb-retrieve___Retrieve"

async with httpx.AsyncClient(auth=sigv4) as http_client:
    async with streamable_http_client(gw_url, http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as mcp_session:
            await mcp_session.initialize()                 # MCP handshake (required)

            listing = await mcp_session.list_tools()       # what an agent sees over MCP
            tool_name = next(
                t.name for t in listing.tools if t.name.split("___")[-1] == "Retrieve"
            )
            print(f"Gateway tool: {tool_name}")

            result = await mcp_session.call_tool(
                name=tool_name,
                arguments={"retrievalQuery": {"text": "What are Octank's key financial results?"}},
            )
            print(f"isError: {result.isError}")
            print("=== Retrieved via Gateway (ALLOWED) ===")
            print(result.content[0].text[:800] if result.content else result)

## `LOG_ONLY` vs `ENFORCE`

| Mode | Behavior |
|---|---|
| `LOG_ONLY` | Cedar evaluates and **logs** the decision but **does not block**. Use to audit and validate policies before enforcing. |
| `ENFORCE` | Cedar evaluates and **blocks** denied requests. The caller gets a JSON-RPC `-32002` error. Use after validating in `LOG_ONLY`. |

A Cedar policy that actually **blocks** needs two things (both live-verified):

- the Gateway must be in **`ENFORCE`** mode, and
- the policy's **resource must be constrained** to a specific `AgentCore::Gateway`.

```cedar
// forbid overrides the earlier permit; this is what Step 10 adds
forbid(principal, action, resource == AgentCore::Gateway::"<gateway-arn>");
```

## Step 10: Update authorization dynamically: `ENFORCE` + `forbid`

Now change the decision **without recreating anything**: flip the Gateway to `ENFORCE` and add
a `forbid` policy. Cedar's `forbid` always overrides `permit`, so every caller is now denied on
this Gateway. This is the policy-as-code workflow, where authorization changes are just API calls.

In [ ]:
# 1. Flip the gateway LOG_ONLY -> ENFORCE (re-send its current config).
gw_info = ac.get_gateway(gatewayIdentifier=gw_id)
ac.update_gateway(
    gatewayIdentifier=gw_id,
    name=gw_info["name"],
    roleArn=gw_info["roleArn"],
    protocolType="MCP",
    authorizerType="AWS_IAM",
    policyEngineConfiguration={"arn": pe_arn, "mode": "ENFORCE"},
)
for _ in range(24):
    if ac.get_gateway(gatewayIdentifier=gw_id)["status"] == "READY":
        break
    time.sleep(5)
print("Gateway mode: ENFORCE")

# 2. Add the forbid policy (unique name per gateway to avoid re-run collisions).
forbid_statement = (
    f'forbid(\n'
    f'  principal,\n'
    f'  action,\n'
    f'  resource == AgentCore::Gateway::"{gw_arn}"\n'
    f');'
)
print(f"Cedar forbid policy:\n{forbid_statement}\n")

deny_response = ac.create_policy(
    policyEngineId=pe_id,
    name=f"deny_gateway_{gw_id.split('-')[-1]}",
    definition={"cedar": {"statement": forbid_statement}},
    validationMode="IGNORE_ALL_FINDINGS",
)
deny_policy_id = deny_response["policyId"]
for _ in range(12):
    st = ac.get_policy(policyEngineId=pe_id, policyId=deny_policy_id)["status"]
    if st in ("ACTIVE", "CREATE_FAILED"):
        break
    time.sleep(5)
print(f"Deny policy: {deny_policy_id} [{st}]")
# Initial settle. Gateway authorization changes are cache-backed and can take a
# while to propagate, so this is just a head start — Step 11 polls/retries until
# the block actually takes effect rather than relying on a fixed wait.
time.sleep(15)

## Step 11: Test denied access (`ENFORCE` + `forbid`)

Same MCP client, same tool. But `ENFORCE` + `forbid` means the Gateway now denies the call
and raises an `McpError` with JSON-RPC code **`-32002`**. We catch it *inside* the session
context, where it surfaces directly (once it unwinds through the client's task group on context
exit it gets wrapped in nested `ExceptionGroup`s, which are awkward to unpack). The forbid also
hides the tool from `tools/list`, so we call it by the name resolved in Step 9.

**On propagation:** Gateway authorization changes are cache-backed and can take a while to
take effect (AWS notes up to about 15 minutes in the worst case). So instead of a single call
after a fixed sleep, we **retry**: re-issue the call until it's blocked, backing off between
attempts. In practice the block lands within seconds, but this keeps the test robust when it
doesn't.

In [ ]:
import asyncio

async def call_once():
    """Issue one retrieve through the gateway. Returns ('blocked'|'allowed'|('error',code), detail)."""
    async with httpx.AsyncClient(auth=sigv4) as http_client:
        async with streamable_http_client(gw_url, http_client=http_client) as (read, write, _):
            async with ClientSession(read, write) as mcp_session:
                await mcp_session.initialize()
                try:
                    result = await mcp_session.call_tool(
                        name=tool_name,
                        arguments={"retrievalQuery": {"text": "What are Octank's key financial results?"}},
                    )
                    return "allowed", (result.content[0].text[:400] if result.content else str(result))
                except McpError as e:
                    if e.error.code == -32002:
                        return "blocked", e.error.message
                    return ("error", e.error.code), e.error.message

# Retry until the forbid propagates (cache-backed; usually seconds, allow generously).
MAX_ATTEMPTS, DELAY = 20, 15   # up to ~5 min total
outcome, detail = None, None
for attempt in range(1, MAX_ATTEMPTS + 1):
    outcome, detail = await call_once()
    if outcome == "blocked":
        print(f"BLOCKED by Cedar policy enforcement (attempt {attempt})")
        print(f"   {detail}")
        break
    if isinstance(outcome, tuple):   # some other MCP error — stop and surface it
        print(f"Unexpected MCP error [{outcome[1]}]: {detail}")
        break
    print(f"  attempt {attempt}: still ALLOWED — forbid not propagated yet, retrying in {DELAY}s...")
    await asyncio.sleep(DELAY)
else:
    print(f"Still ALLOWED after {MAX_ATTEMPTS} attempts — policy did not propagate in time.")
    print(f"   last response: {detail}")

## Step 12: Cleanup

Delete in dependency order: **policies → target → gateway → policy engine → KB → gateway IAM role**.
Each step is wrapped so one failure doesn't block the rest.

> **Guarded, off by default.** This cell only deletes when `RUN_CLEANUP = True`. Leave it
> `False` to keep every deployed resource; flip it to `True` and re-run once you're done
> experimenting.

In [ ]:
# Flip to True to tear everything down. Left False so the deployed resources persist.
RUN_CLEANUP = False

if not RUN_CLEANUP:
    print("RUN_CLEANUP is False — skipping deletion. All resources are left in place.")
    print("Set RUN_CLEANUP = True and re-run this cell to tear everything down.")
else:
    # Cedar policies
    for pid in (permit_policy_id, deny_policy_id):
        try:
            ac.delete_policy(policyEngineId=pe_id, policyId=pid)
            print(f"Deleted policy: {pid}")
        except Exception as e:
            print(f"Policy {pid} cleanup: {e}")

    # Target -> gateway -> policy engine
    try:
        ac.delete_gateway_target(gatewayIdentifier=gw_id, targetId=target_id)
        print(f"Deleted target: {target_id}")
        time.sleep(3)
    except Exception as e:
        print(f"Target cleanup: {e}")

    try:
        ac.delete_gateway(gatewayIdentifier=gw_id)
        print(f"Deleted gateway: {gw_id}")
        time.sleep(3)
    except Exception as e:
        print(f"Gateway cleanup: {e}")

    try:
        ac.delete_policy_engine(policyEngineId=pe_id)
        print(f"Deleted policy engine: {pe_id}")
    except Exception as e:
        print(f"Policy engine cleanup: {e}")

    # KB + data source + KB execution role + S3 bucket
    kb.delete_kb(delete_iam=True, delete_s3_bucket=True)

    # Gateway IAM role (auto-created in Step 4)
    try:
        iam.detach_role_policy(
            RoleName=gateway_role_name,
            PolicyArn=f"arn:aws:iam::{account_id}:policy/{gateway_policy_name}",
        )
        iam.delete_policy(PolicyArn=f"arn:aws:iam::{account_id}:policy/{gateway_policy_name}")
        iam.delete_role(RoleName=gateway_role_name)
        print(f"Deleted gateway role: {gateway_role_name}")
    except Exception as e:
        print(f"Gateway role cleanup: {e}")

    print("\nCleanup complete.")

## Summary

You added **policy-as-code authorization** to a BMKB + AgentCore Gateway using a Cedar Policy Engine:

| Layer | Component | What it does |
|-------|-----------|--------------|
| **Knowledge Base** | Bedrock Managed KB | Fully managed vector store and ingestion |
| **Access** | AgentCore Gateway (MCP) | Centralized tool server; agents never see the KB ID |
| **Auth (identity)** | IAM + SigV4 | Gateway authenticates the caller |
| **Auth (policy)** | Cedar Policy Engine | Per-request permit or forbid, evaluated on every call |

### What you demonstrated

- **`LOG_ONLY`**: Cedar evaluates and logs but never blocks (safe validation)
- **`permit`**: an authorized caller retrieves through the Gateway
- **`ENFORCE` + `forbid`**: flipping mode and adding a policy blocks the same caller (`-32002`), with **no resource recreation**
- **Resource-level control**: Cedar governs *whether* the Gateway can be invoked. It does not filter *which* documents come back

### What's next?

- **Per-principal policies.** Constrain `principal` (not just `resource`) to allow some callers and deny others
- **Metadata filters.** Combine Cedar (gateway-level) with document-level scoping ([Pattern 2: Metadata Filters](../04-security-and-access-controls/notebooks/02-metadata-filters.ipynb))
- **JWT auth.** Cognito or an external IdP for token-based Gateway auth (`05-gateway-jwt-cognito.ipynb`, `06-gateway-jwt-cedar.ipynb`)
- **Full governance.** JWT + Cedar + interceptor + metadata filters together (`08-full-governance.ipynb`)